In [0]:
# MAGIC %pip install confluent-kafka

import sys
import os
import requests
import json
import logging
from confluent_kafka import Producer

sys.path.append(os.path.abspath('.'))

dbutils.widgets.text("KAFKA_KEY", "")
dbutils.widgets.text("KAFKA_SECRET", "")

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("FlightProducer")

try:
    from config import KAFKA_BOOTSTRAP_SERVERS, KAFKA_TOPIC, KAFKA_API_KEY, KAFKA_API_SECRET
    logger.info("Configuration loaded successfully.")
except ImportError:
    KAFKA_API_KEY = dbutils.widgets.get("KAFKA_KEY")
    KAFKA_API_SECRET = dbutils.widgets.get("KAFKA_SECRET")
    from config import KAFKA_BOOTSTRAP_SERVERS, KAFKA_TOPIC

def get_producer():
    conf = {
        'bootstrap.servers': KAFKA_BOOTSTRAP_SERVERS,
        'security.protocol': 'SASL_SSL',
        'sasl.mechanisms': 'PLAIN',
        'sasl.username': KAFKA_API_KEY,
        'sasl.password': KAFKA_API_SECRET,
        'client.id': 'databricks-producer'
    }
    return Producer(conf)

def fetch_and_send():
    """Виконує ОДИН цикл завантаження та відправки даних"""
    producer = get_producer()
    url = "https://opensky-network.org/api/states/all"
    
    logger.info(f"Starting ingestion task for topic: {KAFKA_TOPIC}")
    
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        states = response.json().get("states", [])
        
        if not states:
            logger.warning("No flight data received from API.")
            return

        batch = states[:100]
        for s in batch:
            payload = {
                "icao24": s[0],
                "callsign": s[1].strip() if s[1] else None,
                "origin_country": s[2],
                "longitude": s[5],
                "latitude": s[6],
                "altitude": s[7],
                "on_ground": s[8],
                "velocity": s[9]
            }
            producer.produce(
                KAFKA_TOPIC, 
                json.dumps(payload).encode('utf-8')
            )
        
        producer.flush()
        logger.info(f"Successfully sent {len(batch)} flights to Kafka. Task complete.")
        
    except Exception as e:
        logger.error(f"Critical error during ingestion: {e}")
        sys.exit(1) 

if __name__ == "__main__":
    if not KAFKA_API_KEY or not KAFKA_API_SECRET:
        logger.error("KAFKA_KEY or KAFKA_SECRET is missing!")
        sys.exit(1)
    else:
        fetch_and_send()